<a href="https://colab.research.google.com/github/scshay/DataScience_Shay/blob/main/Generation%20Brasil/Desafios%20t%C3%A9cnicos/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parte 1
Não há código envolvido, somente etapas de preparação dos programas utilizados: Google Sheets API, Google Drive API e o próprio Google Colab

# Parte 2

Célula 01 · Instalar

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

Célula 02 · Importações e Autenticação (service account)

In [3]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@gen-lang-client-0686829899.iam.gserviceaccount.com


Célula 03 · Abrir a planilha

In [6]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1Thl9ix1fjgA-cL2YS7eV6kSpJlxSiyOWz7l8DDTo-Rw"
NOME_TABELA = "pilotos"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Telemetria_Esportiva | pilotos | https://docs.google.com/spreadsheets/d/1Thl9ix1fjgA-cL2YS7eV6kSpJlxSiyOWz7l8DDTo-Rw


Célula 04 · Cabeçalho – Tabela Dimensão

In [8]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["piloto_id", "nome_ficticio", "equipe","categoria","pais"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


Célula 05 · 10 linhas de teste – Tabela Dimensão

In [14]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados

    # Gera a palavra aleatória em latim capitalizada (ex: Impedit, Tempore)
    palavra_aleatoria = fake.word().capitalize()

    return {
       "piloto_id": i,
       "nome_ficticio": f"Piloto {fake.first_name()}", # Gera nomes de pessoas reais,
       "equipe": f"Equipe {palavra_aleatoria} Racing",
       "categoria": random.choice(["formula_sim", "kart", "rally_sim","stock_sim"]),
       "pais": fake.country(),   # gera nomes de países de verdade
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


CÉLULA 6 · Conferência – Tabela Dimensão

In [10]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['piloto_id', 'nome_ficticio', 'equipe', 'categoria', 'pais'], ['1', 'Piloto Henry Gabriel', 'Equipe Tempore Racing', 'formula_sim', 'Angola'], ['2', 'Piloto Samuel', 'Equipe Harum Racing', 'formula_sim', 'Costa do Marfim']]


# Parte 3

CÉLULA 7 · Funções de envio de dados - Planilha

In [15]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""
    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""
    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))
    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])
    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto
    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""
    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

CÉLULA 8 · Carregar Bases de dados - Prontas

In [16]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
# Altere o nome da tabela de acordo com a sua Base de dados
BASES = {
    "pilotos.csv": pd.read_csv("pilotos.csv"),
    "voltas.csv": pd.read_csv("voltas.csv"),
}
# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving voltas.csv to voltas.csv
Saving pilotos.csv to pilotos.csv
{'pilotos.csv': 10800, 'voltas.csv': 55000} | total: 65800


CÉLULA 9 · Enviar

In [17]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  pilotos.csv: 5.000 / 10.800
  pilotos.csv: 10.000 / 10.800
  pilotos.csv: 10.800 / 10.800
OK pilotos.csv: 10.800 linhas na planilha
  voltas.csv: 5.000 / 55.000
  voltas.csv: 10.000 / 55.000
  voltas.csv: 15.000 / 55.000
  voltas.csv: 20.000 / 55.000
  voltas.csv: 25.000 / 55.000
  voltas.csv: 30.000 / 55.000
  voltas.csv: 35.000 / 55.000
  voltas.csv: 40.000 / 55.000
  voltas.csv: 45.000 / 55.000
  voltas.csv: 50.000 / 55.000
  voltas.csv: 55.000 / 55.000
OK voltas.csv: 55.000 linhas na planilha


CÉLULA 10 · Validar

In [18]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 65.800
A planilha confere com a base gerada.
